# Step 0.1 - Chemical Checker bioactivity signatures for JUMP compounds

**What this notebook does.** It builds one table that describes every JUMP Cell Painting compound
by its *Chemical Checker* bioactivity signature and saves it as a Parquet file for later steps.

**Background.** The [Chemical Checker](https://chemicalchecker.org) (CC) summarizes what is known
about a small molecule as 25 numerical "signature spaces", grouped into five levels:

| Level | What it describes | Spaces |
|---|---|---|
| A - Chemistry | structure, scaffolds, physicochemical properties | A1–A5 |
| B - Targets | proteins the molecule binds or modulates | B1–B5 |
| C - Networks | pathways and biological networks reached by those targets | C1–C5 |
| D - Cells | effects in cell-based assays (e.g. gene expression, growth inhibition) | D1–D5 |
| E - Clinics | therapeutic uses, indications and side effects | E1–E5 |

Each space is a vector of 128 numbers, so all 25 spaces together give 3,200 numbers per compound.
Two versions are used:

* **Signature type 2 (sign2)** is derived from experimental data, so it only exists where data exist.
* **Signature type 3 (sign3)** is predicted by a neural network for *any* molecule, so every compound
  has all 25 spaces.

In this project the CC signatures serve as a reference for *what a compound does biologically*,
against which the image-based representations are compared.

**Inputs** (Google Cloud Storage)
* CC sign3 HDF5 files, one per space (`.../Chemical_Checker_Embeddings/sign3_signatures/*.h5`)
* CC sign2 Parquet files, one per level (`.../sign2_signatures/Block_[A-E]_signatures.parquet`)
* JUMP compound metadata with standardized SMILES and InChIKeys (`JUMP_CP_compound_metadata_with_SMILES_standard.csv`)

**Outputs**
* `JUMPCP_Sign3_CC_final.parquet` - sign3, 3,200 dimensions plus metadata
* `JUMPCP_CC_Sign2_Global_Signatures.parquet` - sign2, 3,200 dimensions (with gaps)

**Where to run it.** Written for Google Colab / Colab Enterprise: it uses the `gsutil` command-line tool and
caches immutable downloads under the repository-level `data/raw/` directory and retains assembled local
outputs under this stage's `results/` directory. You need read/write access to the
`gs://calico-reference-embeddings` Cloud Storage bucket for the remote sources and uploads.

> **Note on the sign2 output path.** This notebook writes sign2 to
> `Annotations_from_Roman/Bayer_data_harmonized/annotations_from_Roman/`, but later notebooks read it
> from `embeddings/Chemical_Checker_Embeddings/parquets/`. The file was copied between these
> locations outside this notebook.

## Setup

All libraries used in this notebook are imported in the next cell. Locally they are installed by `uv sync`
(see the main README); on Google Colab, first run
`!pip install git+https://github.com/axiomcura/ModelBenchmarking.git`, which installs the same dependencies.

In [ ]:
import gc
import glob
import os
import re
import shutil
import subprocess
import time

import duckdb
import h5py
import numpy as np
import pandas as pd
from IPython.display import display

from model_benchmarking.paths import cache_gcs_file, notebook_artifact_dirs

RAW_DATA_DIR, RESULTS_DIR = notebook_artifact_dirs("0.download_and_assemble", "chemical_checker")
METADATA_RAW_DIR, _ = notebook_artifact_dirs("0.download_and_assemble", "compound_metadata")
SIGN3_RESULTS_DIR = RESULTS_DIR / "sign3_spaces"
SIGN3_RESULTS_DIR.mkdir(parents=True, exist_ok=True)


## Part 1 - Convert each sign3 space from HDF5 to Parquet

The Chemical Checker distributes each signature space as an HDF5 file holding a matrix `V`
(molecules × 128) and the matching molecule identifiers (`keys`, which are InChIKeys).

### 1.1 Locate the input files
Lists the 25 sign3 HDF5 files in Cloud Storage and creates a local working folder.

In [ ]:
gcs_input_dir = 'gs://calico-reference-embeddings/embeddings/Chemical_Checker_Embeddings/sign3_signatures'
gcs_output_dir = 'gs://calico-reference-embeddings/embeddings/Chemical_Checker_Embeddings/sign3_parquets'

local_h5_dir = RAW_DATA_DIR / 'sign3_h5'
os.makedirs(local_h5_dir, exist_ok=True)

print("🔍 Scanning GCS bucket for .h5 files...")
ls_output = subprocess.check_output(f"gsutil ls {gcs_input_dir}/*.h5", shell=True, text=True)
all_gcs_files =[f.strip() for f in ls_output.strip().split('\n') if f.strip()]

print(f"✅ Found {len(all_gcs_files)} .h5 files to process.")

### 1.2 Convert one space at a time
For each space: download it, name the columns `<space>_Dim_1 … <space>_Dim_128`, add the `InChIKey` column, write Parquet locally under `results/sign3_spaces/`, and upload the same file. Raw HDF5 downloads are retained under `data/raw/chemical_checker/`. Only one space is held in memory at a time, and spaces present both locally and in GCS are skipped, so the cell can be re-run after an interruption.

In [ ]:
for gcs_file in all_gcs_files:
    filename = os.path.basename(gcs_file)

    # Safely extract space name (e.g. 'A1')
    match = re.search(r'([A-E][1-5])', filename.upper())
    if not match:
        continue
    space_name = match.group(1)

    gcs_parquet = f"{gcs_output_dir}/{space_name}_sign3.parquet"
    local_parquet = SIGN3_RESULTS_DIR / f"{space_name}_sign3.parquet"
    local_h5 = f"{local_h5_dir}/{filename}"

    # SAFETY CHECK: Skip if already converted
    check_exists = subprocess.run(f"gsutil -q stat {gcs_parquet}", shell=True)
    if check_exists.returncode == 0 and local_parquet.exists():
        print(f"⏭️ {space_name} already exists locally and in Cloud Storage! Skipping...")
        continue

    print(f"\n🚀 PROCESSING SPACE {space_name}...")

    # 1. Download just THIS ONE file
    print(f"   📥 Downloading {filename}...")
    os.system(f"gsutil cp {gcs_file} {local_h5}")

    # 2. Extract Data (Using numpy string conversion is much faster/lighter than list comprehensions)
    print(f"   🧠 Loading into RAM...")
    with h5py.File(local_h5, 'r') as f:
        data_v = f['V'][()].astype(np.float32)
        # .astype(str) handles byte decoding safely and efficiently
        keys = np.array(f['keys']).astype(str)

    num_cols = data_v.shape[1]
    col_names = [f"{space_name}_Dim_{i+1}" for i in range(num_cols)]

    # 3. Build DataFrame and Save
    print(f"   💾 Saving to local Parquet...")
    df = pd.DataFrame(data_v, columns=col_names)
    df.insert(0, 'InChIKey', keys)
    df.to_parquet(local_parquet, compression='snappy', index=False)

    # 4. Upload to GCS
    print(f"   📤 Uploading {space_name} to Google Cloud Storage...")
    os.system(f"gsutil cp {local_parquet} {gcs_parquet}")

    # 5. Aggressive Cleanup
    print("   📦 Keeping raw input and local result; releasing RAM...")
    # Keep the raw HDF5 input and local Parquet result in their managed directories.
    del data_v, keys, df
    gc.collect()

print("\n🎉 ALL SPACES SUCCESSFULLY CONVERTED TO PARQUET!")

## Part 2 - Merge the 25 sign3 spaces into one table

Starting from the JUMP compound metadata, the 25 spaces are joined one after another on `InChIKey`. The joins
are *inner* joins, so a compound is kept only if it appears in every space: every retained compound has all
3,200 dimensions and no missing values.

### 2.1 Configuration
Cloud Storage locations and local working files.

In [ ]:
metadata_csv = cache_gcs_file(
    'gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/annotations_from_Roman/JUMP_CP_compound_metadata_with_SMILES_standard.csv',
    METADATA_RAW_DIR / 'JUMP_CP_compound_metadata_with_SMILES_standard.csv',
)
gcs_parquet_dir = 'gs://calico-reference-embeddings/embeddings/Chemical_Checker_Embeddings/sign3_parquets'
gcs_final_output = 'gs://calico-reference-embeddings/embeddings/Chemical_Checker_Embeddings/parquets/JUMPCP_Sign3_CC_final.parquet'

local_parquet_dir = SIGN3_RESULTS_DIR
master_accum_file = RESULTS_DIR / 'JUMPCP_Sign3_CC_final.parquet'
temp_accum_file = RESULTS_DIR / '_tmp' / 'temp_accumulator.parquet'
duckdb_temp_dir = RESULTS_DIR / '_tmp' / 'duckdb_temp'

os.makedirs(local_parquet_dir, exist_ok=True)
os.makedirs(duckdb_temp_dir, exist_ok=True)

### 2.2 Prepare the compound table
One row per unique InChIKey with all metadata columns; text columns are stored as strings to prevent mixed-type errors when writing Parquet.

In [ ]:
print("📂 Loading full metadata from CSV...")
df_csv = pd.read_csv(metadata_csv)

# Auto-detect the InChIKey column
inchikey_col = next((col for col in df_csv.columns if 'inchikey' in col.lower()), None)
if not inchikey_col:
    raise KeyError("❌ Could not find an 'InChIKey' column in your CSV. Please check your headers!")

# 1. Rename it to exactly 'InChIKey' so the DuckDB join logic perfectly aligns
df_csv.rename(columns={inchikey_col: 'InChIKey'}, inplace=True)

# 2. Drop rows missing the key, strip whitespace, and deduplicate
# (Deduplication prevents exponential row explosion during joins!)
df_csv = df_csv.dropna(subset=['InChIKey'])
df_csv['InChIKey'] = df_csv['InChIKey'].astype(str).str.strip()
df_csv = df_csv.drop_duplicates(subset=['InChIKey'], keep='first')

# 3. Clean mixed data types to prevent PyArrow compression crashes
object_columns = df_csv.select_dtypes(include=['object']).columns
for col in object_columns:
    df_csv[col] = df_csv[col].astype(str)

print(f"✅ Extracted {len(df_csv):,} unique target molecules with ALL metadata columns.")

# Save this rich metadata table as our starting "Master File"
df_csv.to_parquet(master_accum_file, compression='snappy', index=False)
del df_csv # Free up RAM!

### 2.3 Download the 25 sign3 Parquet files

In [ ]:
print("\n📥 Pulling 25 Parquets from GCS to local SSD... (Fast multithreaded download)")
os.system(f"gsutil -m cp {gcs_parquet_dir}/*_sign3.parquet {local_parquet_dir}/")

parquet_spaces = sorted(glob.glob(f"{local_parquet_dir}/*_sign3.parquet"))
print(f"   ✅ Downloaded {len(parquet_spaces)} files for assembly.")

if len(parquet_spaces) == 0:
    raise FileNotFoundError("No parquet files found! Check your GCS path.")

### 2.4 Join the spaces one by one
[DuckDB](https://duckdb.org) performs each join on disk ("out-of-core"), which avoids running out of memory with more than 100,000 compounds × 3,200 columns.

In [ ]:
print("\n🚀 Starting Incremental 2-Way INNER Joins...")
overall_start_time = time.time()

con = duckdb.connect()
con.execute("PRAGMA memory_limit='4GB'") # Force early SSD spilling to avoid pinning errors
con.execute(f"PRAGMA temp_directory='{duckdb_temp_dir}'")
con.execute("PRAGMA preserve_insertion_order=false")

for i, f in enumerate(parquet_spaces):
    space_name = os.path.basename(f).replace('_sign3.parquet', '')
    print(f"   [{i+1}/{len(parquet_spaces)}] Merging Space {space_name}...")

    step_start = time.time()

    # 🚨 m.* carries ALL of your metadata columns forward through every single loop!
    query = f"""
    COPY (
        SELECT
            m.*,
            r.* EXCLUDE (InChIKey)
        FROM read_parquet('{master_accum_file}') m
        INNER JOIN read_parquet('{f}') r
        ON m.InChIKey = r.InChIKey
    ) TO '{temp_accum_file}' (FORMAT PARQUET, COMPRESSION 'snappy');
    """

    try:
        con.execute(query)
    except Exception as e:
        con.close()
        raise RuntimeError(f"DuckDB Merge Error on {space_name}: {e}")

    # Atomically replace the old master with the newly merged file
    os.replace(temp_accum_file, master_accum_file)

    current_rows = con.execute(f"SELECT COUNT(*) FROM read_parquet('{master_accum_file}')").fetchone()[0]
    print(f"      -> Done in {time.time() - step_start:.1f}s | {current_rows:,} molecules remain.")

con.close()
print(f"\n✅ All 25 spaces successfully merged in {time.time() - overall_start_time:.1f} seconds!")

### 2.5 Check, upload and clean up

In [ ]:
print("\n📊 Verifying the dense assembled dataset...")
df_assembled = pd.read_parquet(master_accum_file)

print("-" * 65)
print(f"   Molecules (Rows):     {df_assembled.shape[0]:,} (Dense Data Only!)")
print(f"   Dimensions (Columns): {df_assembled.shape[1]:,}")
print("-" * 65)

# Show a quick preview to confirm metadata + embeddings are there
display(df_assembled.head(3))

print("\n📤 Uploading fully assembled annotated dataset to Google Cloud Storage...")
os.system(f"gsutil cp {master_accum_file} {gcs_final_output}")

print(f"🎉 SUCCESS! Annotated Global Signatures saved permanently to:")
print(f"   {gcs_final_output}")

print("\n🧹 Removing temporary DuckDB files; retaining raw data and results...")
shutil.rmtree(duckdb_temp_dir, ignore_errors=True)
print(f"Local assembled result retained at: {master_accum_file}")

print("Done! You are ready to analyze the data.")

## Part 3 - Build the sign2 table

The sign2 signatures were already stored as five Parquet files, one per Chemical Checker level (Blocks A–E,
each holding the five spaces of that level). Sign2 is **sparse**: many compounds lack data for some levels.

### 3.1 Configuration

In [ ]:
csv_path = cache_gcs_file(
    'gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/annotations_from_Roman/JUMP_CP_compound_metadata_with_SMILES_standard.csv',
    METADATA_RAW_DIR / 'JUMP_CP_compound_metadata_with_SMILES_standard.csv',
)

# Output location requested
gcs_output_parquet = 'gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/annotations_from_Roman/JUMPCP_CC_Sign2_Global_Signatures.parquet'

local_parquet_dir = RAW_DATA_DIR / 'sign2_blocks'
local_output_parquet = RESULTS_DIR / 'JUMPCP_CC_Sign2_Global_Signatures.parquet'
duckdb_temp_dir = RESULTS_DIR / '_tmp' / 'duckdb_temp'

os.makedirs(local_parquet_dir, exist_ok=True)
os.makedirs(duckdb_temp_dir, exist_ok=True)

# Clean up any leftover files from previous runs
if os.path.exists(local_output_parquet): os.remove(local_output_parquet)

### 3.2 Load the JUMP compound identifiers

In [ ]:
print(f"📂 Loading target subset from {csv_path}...")
df_csv = pd.read_csv(csv_path)

# Auto-detect the InChIKey column (e.g., 'Metadata_InChIKey', 'InChIKey')
inchikey_col = next((col for col in df_csv.columns if 'inchikey' in col.lower()), None)
if not inchikey_col:
    raise KeyError("❌ Could not find an 'InChIKey' column in your CSV. Please check your headers!")

# Extract unique target keys
target_inchikeys = df_csv[inchikey_col].dropna().astype(str).str.strip().unique()
print(f"✅ Extracted {len(target_inchikeys):,} unique target InChIKeys.")

# Wrap in a lightweight DataFrame for DuckDB to use as the Driving Table
df_targets = pd.DataFrame({'Target_InChIKey': target_inchikeys})

### 3.3 Download the five sign2 blocks

In [ ]:
print("\n📥 Pulling Sign2 Parquets from GCS to local SSD... (Fast multithreaded download)")
# We pull all 5 blocks at once to the local disk to make the SQL join lightning fast
os.system(f"gsutil -m cp gs://calico-reference-embeddings/embeddings/Chemical_Checker_Embeddings/sign2_signatures/Block_*.parquet {local_parquet_dir}/")

# Map the local paths
blk_A = f"{local_parquet_dir}/Block_A_signatures.parquet"
blk_B = f"{local_parquet_dir}/Block_B_signatures.parquet"
blk_C = f"{local_parquet_dir}/Block_C_signatures.parquet"
blk_D = f"{local_parquet_dir}/Block_D_signatures.parquet"
blk_E = f"{local_parquet_dir}/Block_E_signatures.parquet"

### 3.4 Join the blocks
Compounds must be in the JUMP list **and** in Block A (inner join); Blocks B–E are attached with *left* joins, so a compound without data at a level gets empty (NaN) columns.

In [ ]:
print("\n⚙️ Executing the out-of-core streaming join...")
start_time = time.time()

con = duckdb.connect()
con.execute("PRAGMA memory_limit='8GB'")
con.execute(f"PRAGMA temp_directory='{duckdb_temp_dir}'")

# The SQL Logic:
# 1. INNER JOIN between your CSV targets and Block A creates the rigid intersection.
# 2. SEQUENTIAL LEFT JOINS pull in B, C, D, and E. If a molecule exists in A but
#    is missing in C, the columns for C are safely populated with NULLs (NaNs).
# 3. EXCLUDE (InChIKey) prevents Pandas from creating duplicate columns (InChIKey_1, etc.)

query = f"""
COPY (
    SELECT
        a.*,
        b.* EXCLUDE (InChIKey),
        c.* EXCLUDE (InChIKey),
        d.* EXCLUDE (InChIKey),
        e.* EXCLUDE (InChIKey)
    FROM df_targets t
    INNER JOIN read_parquet('{blk_A}') a ON t.Target_InChIKey = a.InChIKey
    LEFT JOIN read_parquet('{blk_B}') b ON a.InChIKey = b.InChIKey
    LEFT JOIN read_parquet('{blk_C}') c ON a.InChIKey = c.InChIKey
    LEFT JOIN read_parquet('{blk_D}') d ON a.InChIKey = d.InChIKey
    LEFT JOIN read_parquet('{blk_E}') e ON a.InChIKey = e.InChIKey
) TO '{local_output_parquet}' (FORMAT PARQUET, COMPRESSION 'snappy');
"""

try:
    con.execute(query)
    print(f"  ✅ Join completed flawlessly in {time.time() - start_time:.1f} seconds!")
except Exception as e:
    raise RuntimeError(f"DuckDB Merge Error: {e}")
finally:
    con.close()

### 3.5 Upload and check the result

In [ ]:
print("\n📤 Uploading final assembled dataset to Google Cloud Storage...")
os.system(f"gsutil cp {local_output_parquet} {gcs_output_parquet}")

print(f"🎉 SUCCESS! Global Signatures saved permanently to:")
print(f"   {gcs_output_parquet}")

# Load it back to verify the exact intersection numbers
df_final = pd.read_parquet(local_output_parquet)
print("\n" + "=" * 65)
print(f"📊 FINAL SUBSET STATISTICS:")
print(f"   Original Target Molecules: {len(target_inchikeys):,}")
print(f"   Intersected Molecules:     {len(df_final):,}")
print(f"   Total Dimensions (A-E):    {df_final.shape[1] - 1:,} (Excluding InChIKey)")
print("=" * 65)
display(df_final.head())

# Clean up local workspace to save space
print("\n🧹 Removing temporary DuckDB files; retaining raw data and results...")
shutil.rmtree(duckdb_temp_dir, ignore_errors=True)
print(f"Local assembled result retained at: {local_output_parquet}")